# GPU 长什么样：lspci 与 PCIe 拓扑

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/lab-02-gpu-on-pcie.ipynb)

从系统总线视角看 GPU。用 lspci、nvidia-smi -q 和 /proc/cpuinfo 把 GPU 和它在主板上的位置、NUMA 归属联系起来，为后面理解多卡、PCIe 带宽和拓扑调度打基础。

本 notebook 配合书中章节 [GPU 长什么样：lspci 与 PCIe 拓扑](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应书中一个 Step；Markdown 单元摘录了必要的解释，完整上下文请回到书中阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## Why this problem matters

[Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 里 `nvidia-smi` 显示了一行 `00000000:00:04.0`，那是 GPU 的 **PCIe 总线地址**。在单卡 Colab 里它只是个数字，但在多卡服务器里，GPU 挂在哪个 NUMA 节点、走哪条 PCIe Switch、和谁共享带宽，直接决定了训练和推理的性能。Kubernetes 的拓扑调度、HAMi 的设备分配，本质上都在围绕这些信息做决策。本 Lab 用最小命令把这些概念落地。

## Step 1：用 lspci 看总线

In [ ]:
!lspci | grep -i nvidia

预期输出类似：

```text
00:04.0 3D controller: NVIDIA Corporation TU104GL [Tesla T4] (rev a1)
```

读法：

- `00:04.0` = **总线 : 设备 . 功能**（bus:device.function），和 `nvidia-smi` 里的 `00:04.0` 对应；
- `3D controller` = 设备类别（数据中心卡常显示为 `3D controller` 而非 `VGA compatible controller`，因为它没有显示输出）；
- `TU104GL` = 芯片代号（Turing）。

## Step 2：用 nvidia-smi 查 PCIe 链路

In [ ]:
!nvidia-smi --query-gpu=pci.bus_id,pcie.link.gen.max,pcie.link.gen.current,pcie.link.width.max,pcie.link.width.current --format=csv

预期看到：

```text
pci.bus_id, pcie.link.gen.max, pcie.link.gen.current, pcie.link.width.max, pcie.link.width.current
00000000:00:04.0, 3, 1, 16, 16
```

关键字段：

- `pcie.link.gen.max: 3` = 设备支持的最高代际 PCIe **Gen3**（T4 上限）；
- `pcie.link.gen.current: 1` = **当前**运行在 Gen1。这不是故障：GPU 空载时会把链路降速省电，有数据传输时自动升回 Gen3；
- `pcie.link.width.max: 16` = 16 条 lane，即 **x16**。

想看完整的 PCI 小节，可以用 `!nvidia-smi -q -d PCI`。它会把上面这些字段按 `GPU Link Info → PCIe Generation / Link Width` 的层级展开，还会列出 Device Id（T4 是 `0x1EB810DE`，`10DE` 是 NVIDIA 的 vendor id）。

## Step 3：算一下理论带宽

PCIe 是全双工的，每条 lane 单向带宽约：

| 代际 | 单 lane 单向 (GB/s) | x16 单向 (GB/s) | x16 双向合计 (GB/s) |
| :--- | :--- | :--- | :--- |
| Gen3 | ~0.985 | ~15.75 | ~31.5 |
| Gen4 | ~1.969 | ~31.5 | ~63 |
| Gen5 | ~3.938 | ~63 | ~126 |

T4 在 Gen3 x16 下，CPU→GPU 单向理论峰值约 **15.75 GB/s**（实测 `cudaMemcpy` 通常 11–13 GB/s）。对比 T4 的显存带宽 **320 GB/s**：**数据一旦进了显存，访问速度比从 CPU 搬快 20 倍以上**。这就是为什么 GPU 编程要尽量减少“CPU↔GPU 来回搬数据”（[Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 会再遇到）。

## Step 4：NUMA 归属（理解它，跳过实现）

In [ ]:
!lscpu | grep -E "^(Socket|NUMA)"

Colab 通常是单 NUMA / 单 socket（输出 `Socket(s): 1`、`NUMA node(s): 1`）。再跑一下 `!nvidia-smi topo -m`：它会把 GPU 之间、GPU 与 CPU 之间的连接关系画成矩阵，并给出 GPU 的 `CPU Affinity` 与 `NUMA Affinity`。Colab 单卡上这张矩阵只有一格，但在多卡服务器上，不同 GPU 会落到不同 NUMA 节点：

![多卡服务器上的 PCIe / NUMA 拓扑：两个 CPU 各挂一个 PCIe Switch 和两张 GPU，GPU 0 与 GPU 1 之间有 NVLink](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/pcie-numa-topology.svg)

> 这正是 Kubernetes [Device Plugin / DRA 边界](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../foundations/device-plugin-vs-dra/) 里“拓扑约束无法用整数表达”的现实来源：调度器不只关心“有几张卡”，还关心“卡和 CPU 谁更近”。

## Verification

完成后你应该能：

- 用 `lspci` 找到 GPU 的 bus:device.function；
- 把它和 `nvidia-smi` 的 `Bus Id` 对应起来；
- 说出 T4 的 PCIe 代际、链路宽度、理论带宽，以及为什么空载时 `current` 会低于 `max`；
- 解释为什么“数据进显存后比从 CPU 搬快 20 倍”。

## Failure Modes

| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `lspci: command not found` | 镜像没有 `pciutils` | `!apt-get install -y -qq pciutils` |
| `lspci` 里看不到 NVIDIA | Runtime 还是 CPU | 切到 T4 GPU runtime，重连 |
| `pcie.link.gen.current` 是 `1` | 空载省电降速，正常 | 跑一个显存拷贝后再查，会升到 `3` |
| `pcie.link.width.current` 显示 `4` / `8` 而非 `16` | 虚拟化/共享带宽或降速 | 一般不影响学习；多见于被 vGPU/分片的环境 |

## 记录你的 Metadata Card

运行完成后，把实际数值回填到书中该章的 Metadata Card：运行日期、驱动 / CUDA 版本、关键输出数值、遇到的 Failure Mode。欢迎在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提交 PR 或 Issue。